# Permutation test v3 (2/2) — 1000 random-k

Notebook này chạy nhãn thật + **1000 lần random-k** → p_random (chọn gen có hơn chọn ngẫu nhiên k gene không). Hoán vị chạy ở notebook `perm_test_v3_permutation.ipynb`. Hai notebook sinh từ cùng một mã nguồn (cùng fold, pipeline, seed) nên kết quả nhãn thật
giống hệt nhau và ghép được với nhau.

Dữ liệu: `experiment-data-perm/` (tạo bằng `scripts/build_perm_test_data.py`). Notebook chạy **lần lượt mọi bộ trong
`RUN_DATASETS`** trong một lần Run All; muốn chạy một bộ thì để danh sách chỉ còn bộ đó. Nhánh, tác vụ và danh sách k
của từng bộ lấy từ `DATASET_SPECS`.

**Gộp hai notebook:** thêm output của notebook kia làm input và điền thư mục `perm_test_v3_permutation` của nó vào `RESUME_DIRS`
→ notebook này đọc thêm điểm hoán vị/random-k đã có, không chạy lại, và `summary.csv` / `decision.csv` có đủ cả
p_perm lẫn p_random (kết luận cuối cùng).

| Nhánh | Bộ dữ liệu | Tác vụ | k | Permutation / random-k |
|---|---|---|---|---|
| (1) Giai đoạn | `tcga_luad_stage_early_late`, `tcga_lusc_stage_early_late`, `gse68465_stage_early_late` | sớm (I) vs muộn (II–IV), chỉ mẫu u | 10, 50, 100, 200, 500 | 1000 / 1000 |
| (2) Mô học | `tcga_luad_lusc_primary_tumor` (+ cohort GEO khi có, **không gộp cohort**) | LUAD vs LUSC, chỉ mẫu u | 1, 2, 3, 5, 10, 25, 50 | 1000 / 1000 |
| (3) Sanity | `tcga_normal_vs_tumor_sanity` | Normal vs Tumor | 10 | B = 0 (chỉ nhãn thật) |

- **3 pipeline**, chọn gen **trong training fold** và chạy lại ở **mọi hoán vị**: P1 F-test + LR (Zhu & Hastie, 2004) ·
  P2 SVM-RFE + SVM tuyến tính (Guyon et al., 2002) · P3 Nearest Shrunken Centroids/PAM (Tibshirani et al., 2002).
- **Trọng số lớp:** `balanced` = cấu hình **chính**; `none` = **sensitivity analysis**. LR/SVM: `class_weight`;
  PAM: prior bằng nhau (`balanced`) hoặc prior = tỉ lệ lớp (`none`). Không chuẩn hoá dữ liệu (log2 gốc).
- **Fold (giống feature selection):** đọc `folds/fold_<r>/{train,val,test}.h5` của từng bộ — 5 phần P1..P5 theo
  `patient_id`; fold r: test = P_r, val = P_(r+1), train = 3 phần còn lại (bộ LUAD vs LUSC và sanity dùng nguyên fold
  v2). Chọn gen + huấn luyện trên **train**; **p-value tính trên test**; BAcc trên val chỉ ghi lại (`bacc_real_val`),
  không dùng để chọn gì vì mọi tham số cố định trước. Fold cố định cho mọi pipeline, cấu hình, hoán vị và phiên chạy.
- **Hoán vị:** xáo nhãn theo bệnh nhân, seed `(SEED, pid)` → mọi pipeline/cấu hình dùng cùng bộ nhãn xáo (so sánh cặp).
  p = (#null ≥ thật + 1)/(B + 1) (Phipson & Smyth, 2010).
- **Holm** (1979) áp dụng **riêng trong từng họ** = (bộ dữ liệu, tác vụ, cấu hình trọng số, loại kiểm định);
  mỗi họ gồm 3 pipeline × |k| kiểm định.
- **Đầu ra** `perm_test_v3_random_k/<dataset>/`: `fold_assignment.csv`, `real_scores_per_fold.csv`, `real_confusion.csv`,
  `real_selected_genes.csv`, `null_scores_per_fold.csv`, `random_k_scores_per_fold.csv`, `summary.csv`,
  `pvalues_adjusted.csv`, `decision.csv`, `timing.csv`, hình `.png`, `config.json` (dataset – task – k – seed – commit).
  Gộp mọi bộ: `perm_test_v3_random_k/all_decisions.csv`, `all_summary.csv`, `all_pvalues_adjusted.csv`; nén `perm_test_v3_random_k.zip`.
- **Thời gian:** `TIME_BUDGET_H` tính chung cho cả lượt. Sắp hết thì dừng an toàn sau lô đang chạy; phiên sau thêm output
  phiên trước làm input, điền thư mục `perm_test_v3_random_k` cũ vào `RESUME_DIRS` rồi Run All — phần đã xong được bỏ qua.

In [ ]:
# tables: backend cho pd.read_hdf
get_ipython().system("pip install -q tables")

## 1. Cấu hình

In [ ]:
import os, sys, glob, subprocess

# ====== CAU HINH (sua tay) ======
MODE = "random_k"   # notebook nay chi chay phan nay (permutation | random_k); KHONG doi
# Thu tu chay: bo nhanh truoc. De 1 phan tu neu chi muon chay 1 bo.
RUN_DATASETS = ["gse68465_stage_early_late", "tcga_luad_stage_early_late", "tcga_lusc_stage_early_late",
                "tcga_luad_lusc_primary_tumor"]   # sanity (nhanh 3) khong co random-k
PIPELINES = ["ftest_lr", "svm_rfe", "pam"]   # pipeline chay trong phien nay
ALL_PIPELINES = ["ftest_lr", "svm_rfe", "pam"]   # ho kiem dinh day du (de hieu chinh p)
CLASS_WEIGHTS = ["balanced", "none"]       # balanced = cau hinh chinh, none = sensitivity analysis
MAIN_CW = "balanced"
N_SPLITS = 5
SEED = 42
LR_C = 1.0                # LR phat L2 cua P1
SVM_C = 1.0               # SVM tuyen tinh cua P2
ALPHA = 0.05
MIN_GAIN = 0.02           # chenh BAcc toi thieu so voi random-k de coi la "co ich" (dat truoc)
N_JOBS = -1
BATCH_PER_WORKER = 4      # so hoan vi moi worker trong 1 lo; luu ket qua sau moi lo
TIME_BUDGET_H = 11.0      # ngan sach CHUNG cho ca luot (Kaggle gioi han 12h)
GIT_COMMIT = ""           # dan `git rev-parse HEAD` cua repo KLTN; de trong -> thu git, khong co -> "unknown"
DATA_ROOT = ""            # thu muc experiment-data-perm; de trong -> tu tim (vd /kaggle/input/experiment-data-perm)
# Thu muc ket qua cu (chua cac thu muc con <dataset>/) de doc lai, vd
#  - chay tiep phien truoc: ["/kaggle/input/<output-phien-truoc>/perm_test_v3_random_k"]
#  - gop voi notebook kia:  them "/kaggle/input/<output-notebook-kia>/perm_test_v3_permutation"
RESUME_DIRS = []
EXCLUDE_DIAGNOSIS_MISMATCH = True   # nhanh 2: bo ca co primary_diagnosis trai voi project (vd TCGA-43-8118)

K_STAGE = [10, 50, 100, 200, 500]
K_HISTO = [1, 2, 3, 5, 10, 25, 50]
DATASET_SPECS = {
    # nhanh 1: giai doan som vs muon, chi mau u
    "tcga_luad_stage_early_late": {"branch": "1_stage", "task": "early_vs_late", "k": K_STAGE, "n_perm": 1000, "n_random": 1000},
    "tcga_lusc_stage_early_late": {"branch": "1_stage", "task": "early_vs_late", "k": K_STAGE, "n_perm": 1000, "n_random": 1000},
    "gse68465_stage_early_late":  {"branch": "1_stage", "task": "early_vs_late", "k": K_STAGE, "n_perm": 1000, "n_random": 1000},
    # nhanh 2: LUAD vs LUSC tung cohort, khong gop cohort. TCGA: mo hoc trung voi project (nhieu batch) -> tham khao
    "tcga_luad_lusc_primary_tumor": {"branch": "2_histology", "task": "luad_vs_lusc", "k": K_HISTO, "n_perm": 1000, "n_random": 1000},
    # nhanh 3: sanity check, B = 0 (chi nhan that)
    "tcga_normal_vs_tumor_sanity": {"branch": "3_sanity", "task": "normal_vs_tumor", "k": [10], "n_perm": 0, "n_random": 0},
}
# Them cohort GEO cho nhanh 2: thu muc cung dinh dang (expression.h5 key "expression", metadata.csv,
# perm_folds.csv) + 1 dong {"branch": "2_histology", "task": "luad_vs_lusc", "k": K_HISTO, ...} + them vao RUN_DATASETS

assert set(RUN_DATASETS) <= set(DATASET_SPECS), set(RUN_DATASETS) - set(DATASET_SPECS)
assert set(PIPELINES) <= set(ALL_PIPELINES) and MAIN_CW in CLASS_WEIGHTS

IN_KAGGLE = os.path.isdir("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_data_root():
    # Thu muc chua cac bo <dataset>/perm_folds.csv (Kaggle: /kaggle/input/.../experiment-data-perm,
    # khong can manifest.json)
    cands = glob.glob("/kaggle/input/**/perm_folds.csv", recursive=True) + [
        f"{r}/{RUN_DATASETS[0]}/perm_folds.csv" for r in [
            "/content/drive/MyDrive/KLTN/experiment-data-perm", "/content/experiment-data-perm",
            os.path.abspath("../../../experiment-data-perm"), os.path.abspath("experiment-data-perm")]]
    for f in cands:
        root = os.path.dirname(os.path.dirname(f))
        if all(os.path.isfile(os.path.join(root, d, "perm_folds.csv")) for d in RUN_DATASETS):
            return root
    raise FileNotFoundError("Khong tim thay thu muc chua du RUN_DATASETS (<bo>/perm_folds.csv) -> dien DATA_ROOT")


DATA_ROOT = DATA_ROOT or find_data_root()
OUT_ROOT = ("/kaggle/working" if IN_KAGGLE else
            "/content/drive/MyDrive/KLTN/results" if IN_COLAB else os.path.abspath("results"))
RESULT_ROOT = os.path.join(OUT_ROOT, "perm_test_v3_random_k")
os.makedirs(RESULT_ROOT, exist_ok=True)
N_WORKERS = os.cpu_count() if N_JOBS == -1 else N_JOBS

if not GIT_COMMIT:
    try:
        GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                             stderr=subprocess.DEVNULL).strip()
    except Exception:
        GIT_COMMIT = "unknown"
        print("CANH BAO: khong co commit hash -> dien GIT_COMMIT de config.json day du")

print("Che do   :", MODE)
print("Du lieu  :", DATA_ROOT)
print("Ket qua  :", RESULT_ROOT, "| so CPU:", os.cpu_count(), "| commit:", GIT_COMMIT)
print("Pipeline :", PIPELINES, "| class_weight:", CLASS_WEIGHTS, "(chinh:", MAIN_CW + ")")
print("Ngan sach:", TIME_BUDGET_H, "gio cho ca luot | resume:", RESUME_DIRS)
for d in RUN_DATASETS:
    s = DATASET_SPECS[d]
    print(f"  {d:30s} {s['branch']:12s} {s['task']:16s} k={s['k']} | hoan vi {s['n_perm']} | random-k {s['n_random']}")

## 2. Hàm tiện ích: đọc/ghi kết quả, Holm

In [ ]:
import json, time, hashlib, shutil, platform, warnings
from types import SimpleNamespace
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore")
T_START = time.time()


def hours_left():
    return TIME_BUDGET_H - (time.time() - T_START) / 3600


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


DTYPES = {"pid": int, "rep": int, "k": int, "fold": int, "true": int, "pred": int, "count": int, "rank": int,
          "bacc": float, "sec_per_run": float}
FOLD_KEYS = ["class_weight", "pipeline", "k", "fold"]
FILES = {  # ten file -> (khoa bo trung, cot)
    "real": ("real_scores_per_fold.csv", FOLD_KEYS + ["split"], FOLD_KEYS + ["split", "bacc"]),
    "conf": ("real_confusion.csv", ["class_weight", "pipeline", "k", "true", "pred"],
             ["class_weight", "pipeline", "k", "true", "pred", "count"]),
    "gene": ("real_selected_genes.csv", FOLD_KEYS + ["rank"], FOLD_KEYS + ["rank", "gene"]),
    "null": ("null_scores_per_fold.csv", ["pid"] + FOLD_KEYS, ["pid"] + FOLD_KEYS + ["bacc"]),
    "rand": ("random_k_scores_per_fold.csv", ["rep"] + FOLD_KEYS, ["rep"] + FOLD_KEYS + ["bacc"]),
    "time": ("timing.csv", ["pipeline"], ["pipeline", "sec_per_run"]),
}


def typed(d):
    return d.astype({c: t for c, t in DTYPES.items() if c in d.columns})


def load_prev(D, key):
    # Gop file cung ten tu RESUME_DIRS/<dataset> va thu muc ket qua, bo dong trung, ghi lai vao thu muc ket qua
    name, keys, cols = FILES[key]
    path = os.path.join(D.result_dir, name)
    srcs = [os.path.join(r, D.name, name) for r in RESUME_DIRS] + [path]
    frames = [pd.read_csv(f) for f in srcs if os.path.isfile(f) and os.path.getsize(f) > 0]
    if not frames:
        return typed(pd.DataFrame(columns=cols))
    d = typed(pd.concat(frames, ignore_index=True).drop_duplicates(keys, keep="last"))
    d.to_csv(path, index=False)
    return d


def append_csv(rows, D, key):
    if rows:
        path = os.path.join(D.result_dir, FILES[key][0])
        pd.DataFrame(rows).to_csv(path, mode="a", index=False, header=not os.path.exists(path))


def done_ids(df, id_col):
    # {pipeline: tap id da chay du moi cau hinh trong so cua phien nay}
    if not len(df):
        return {}
    d = df[df.class_weight.isin(CLASS_WEIGHTS)].groupby(["pipeline", id_col]).class_weight.nunique()
    d = d[d == len(CLASS_WEIGHTS)].reset_index()
    return d.groupby("pipeline")[id_col].apply(set).to_dict()


def holm(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    adj = np.maximum.accumulate((m - np.arange(m)) * p[o]).clip(max=1)
    out = np.empty(m); out[o] = adj
    return out

## 3. Ba pipeline chọn gen + phân loại

| Mã | Chọn gen (trong training fold) | Phân loại | `balanced` | `none` |
|---|---|---|---|---|
| `ftest_lr` | F-test ANOVA, top-k | LR phạt L2, C = 1 | `class_weight="balanced"` | không trọng số |
| `svm_rfe` | SVM-RFE: loại một nửa số gene mỗi vòng theo w², dừng đúng tại từng k (SVM trong RFE dùng cùng trọng số) | SVM tuyến tính, C = 1 | `class_weight="balanced"` | không trọng số |
| `pam` | Nearest shrunken centroids: ngưỡng Δ để còn đúng k gene, s0 = trung vị s_i | Shrunken centroid | prior bằng nhau | prior = tỉ lệ lớp |

Xếp hạng F-test và thống kê PAM không phụ thuộc trọng số lớp nên tính một lần mỗi fold; SVM-RFE chạy riêng cho từng
cấu hình. SVM dùng kernel tính sẵn `X·Xᵀ` (tương đương `SVC(kernel="linear")`, nhanh hơn khi n ≪ p).

In [ ]:
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

CW_SKLEARN = {"balanced": "balanced", "none": None}


# ---------- P1: F-test + LR ----------
def ftest_rank(X, y):
    F, _ = f_classif(X, y)                                   # gene hang so -> NaN -> xuong cuoi
    return np.argsort(-np.nan_to_num(F, nan=-np.inf), kind="stable")


def make_lr(cw):
    return LogisticRegression(C=LR_C, max_iter=5000, class_weight=CW_SKLEARN[cw])   # L2


# ---------- P2: SVM-RFE + SVM tuyen tinh (Guyon et al., 2002) ----------
class LinearSVM:
    # SVM tuyen tinh 2 lop qua kernel tinh san; giu w de xep hang gene
    def __init__(self, cw):
        self.cw = cw

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        svc = SVC(kernel="precomputed", C=SVM_C, class_weight=CW_SKLEARN[self.cw]).fit(X @ X.T, y)
        self.w = svc.dual_coef_[0] @ X[svc.support_]
        self.b = svc.intercept_[0]
        return self

    def predict(self, X):
        return self.classes_[(X @ self.w + self.b > 0).astype(int)]   # > 0 -> classes_[1]


def svm_rfe_panels(X, y, k_list, cw):
    # Loai mot nua so gene moi vong, dung dung tai moi k -> {k: (gene, SVM da fit tren k gene)}
    targets = sorted(k_list, reverse=True)
    active = np.arange(X.shape[1])
    out = {}
    while True:
        m = LinearSVM(cw).fit(X[:, active], y)
        if len(active) in targets:
            out[len(active)] = (active.copy(), m)
        if len(active) <= targets[-1]:
            break
        n_next = max(len(active) // 2, max(t for t in targets if t < len(active)))
        active = active[np.argsort(-(m.w ** 2), kind="stable")[:n_next]]
    return out


# ---------- P3: Nearest shrunken centroids / PAM (Tibshirani et al., 2002) ----------
class NSCStats:
    # Thong ke cua 1 fold; model(k, cw) = PAM voi nguong Delta giu lai dung k gene
    def __init__(self, X, y):
        self.classes_ = np.unique(y)
        idx = np.searchsorted(self.classes_, y)
        n, K = len(y), len(self.classes_)
        nk = np.bincount(idx)
        self.xbar = X.mean(0)
        self.mu = np.stack([X[idx == j].mean(0) for j in range(K)])
        self.s = np.sqrt(((X - self.mu[idx]) ** 2).sum(0) / max(n - K, 1))     # do lech chuan chung trong lop
        self.s0 = np.median(self.s)
        self.mk = np.sqrt(1.0 / nk - 1.0 / n)
        self.d = (self.mu - self.xbar) / (self.mk[:, None] * (self.s + self.s0))
        self.score = np.abs(self.d).max(0)
        self.order = np.argsort(-self.score, kind="stable")
        self.logprior = {"none": np.log(nk / n), "balanced": np.full(K, -np.log(K))}

    def model(self, k, cw, genes=None):
        if genes is None:            # PAM: nguong Delta de con dung k gene co centroid khac trung binh chung
            sel = self.order[:k]
            delta = self.score[self.order[k]] if k < len(self.order) else 0.0
        else:                        # random-k: gene cho truoc, khong rut gon
            sel, delta = np.asarray(genes), 0.0
        d = self.d[:, sel]
        d_sh = np.sign(d) * np.maximum(np.abs(d) - delta, 0.0)                  # soft-threshold
        scale = self.s[sel] + self.s0
        return _NSCModel(self.classes_, sel, self.xbar[sel] + self.mk[:, None] * scale * d_sh,
                         scale ** 2, self.logprior[cw])


class _NSCModel:
    def __init__(self, classes, sel, mu, scale2, logprior):
        self.classes_, self.sel, self.mu, self.scale2, self.logprior = classes, sel, mu, scale2, logprior

    def predict(self, X):
        Xs = X[:, self.sel]
        d = -(((Xs[:, None, :] - self.mu[None]) ** 2) / self.scale2).sum(2) + 2 * self.logprior
        return self.classes_[d.argmax(1)]


# ---------- 1 fold: 1 pipeline, moi cau hinh trong so, moi k ----------
def fold_predictions(Xtr, ytr, Xte, pipeline, cws, k_list):
    # -> {(cw, k): (du doan tren test, chi so gene da chon)}
    out = {}
    if pipeline == "ftest_lr":
        order = ftest_rank(Xtr, ytr)
        for cw in cws:
            for k in k_list:
                g = order[:k]
                out[(cw, k)] = (make_lr(cw).fit(Xtr[:, g], ytr).predict(Xte[:, g]), g)
    elif pipeline == "svm_rfe":
        for cw in cws:
            for k, (g, m) in svm_rfe_panels(Xtr, ytr, k_list, cw).items():
                out[(cw, k)] = (m.predict(Xte[:, g]), g)
    elif pipeline == "pam":
        st = NSCStats(Xtr, ytr)
        for cw in cws:
            for k in k_list:
                out[(cw, k)] = (st.model(k, cw).predict(Xte), st.order[:k])
    else:
        raise ValueError(pipeline)
    return out


def random_predictions(Xtr, ytr, Xte, genes, pipeline, cw):
    # random-k: k gene ngau nhien + cung bo phan loai cua pipeline
    a, b = Xtr[:, genes], Xte[:, genes]
    if pipeline == "ftest_lr":
        return make_lr(cw).fit(a, ytr).predict(b)
    if pipeline == "svm_rfe":
        return LinearSVM(cw).fit(a, ytr).predict(b)
    if pipeline == "pam":
        return NSCStats(a, ytr).model(None, cw, genes=np.arange(len(genes))).predict(b)
    raise ValueError(pipeline)

## 4. Hàm cho tiến trình con: một hoán vị / một lần random-k

`pid = -1` là nhãn thật; `pid = 0..B-1` là hoán vị. Nhãn của hoán vị `pid` được xáo theo bệnh nhân bằng seed
`(SEED, pid)` → **tất định**: mọi phiên, mọi pipeline và cả hai cấu hình trọng số dùng đúng cùng một bộ nhãn xáo.
Random-k chọn k gene bằng seed `(SEED, 7, rep)`, cùng tập gene cho mọi pipeline và cấu hình. Mỗi dòng kết quả là BAcc
của **một fold**.

In [ ]:
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits


def permute_labels(y, groups, rng):
    # Xao theo benh nhan neu moi benh nhan chi co 1 nhan; nguoc lai xao theo mau
    s = pd.Series(y, index=groups)
    if (s.groupby(level=0).nunique() == 1).all():
        lab = s.groupby(level=0).first()
        mapping = dict(zip(lab.index, rng.permutation(lab.to_numpy())))
        return np.array([mapping[g] for g in groups])
    return rng.permutation(y)


def labels_for(pid, y, groups):
    return y if pid < 0 else permute_labels(y, groups, np.random.default_rng([SEED, pid]))


def run_pid(pid, pipelines, fold_data, folds, y, groups, k_list):
    # -> list dong {pid, class_weight, pipeline, k, fold, bacc}; 1 tien trinh, BLAS 1 luong
    warnings.filterwarnings("ignore")
    yp = labels_for(pid, y, groups)
    rows = []
    with threadpool_limits(1):
        for f, ((Xtr, _, Xte), (tr, _, te)) in enumerate(zip(fold_data, folds), 1):   # fit train, cham test
            for p in pipelines:
                for (cw, k), (pred, _) in fold_predictions(Xtr, yp[tr], Xte, p, CLASS_WEIGHTS, k_list).items():
                    rows.append({"pid": pid, "class_weight": cw, "pipeline": p, "k": k, "fold": f,
                                 "bacc": balanced_accuracy_score(yp[te], pred)})
    return rows


def run_random_rep(rep, pipelines, fold_data, folds, y, k_list):
    warnings.filterwarnings("ignore")
    rng = np.random.default_rng([SEED, 7, rep])
    rows = []
    with threadpool_limits(1):
        for f, ((Xtr, _, Xte), (tr, _, te)) in enumerate(zip(fold_data, folds), 1):   # fit train, cham test
            for k in k_list:
                genes = rng.choice(Xtr.shape[1], size=k, replace=False)
                for p in pipelines:
                    for cw in CLASS_WEIGHTS:
                        pred = random_predictions(Xtr, y[tr], Xte, genes, p, cw)
                        rows.append({"rep": rep, "class_weight": cw, "pipeline": p, "k": k, "fold": f,
                                     "bacc": balanced_accuracy_score(y[te], pred)})
    return rows


def run_chunk(kind, items, fold_data, folds, y, groups, k_list):
    # items: [(pid hoac rep, [pipeline,...])]
    if kind == "null":
        return [r for i, pipes in items for r in run_pid(i, pipes, fold_data, folds, y, groups, k_list)]
    return [r for i, pipes in items for r in run_random_rep(i, pipes, fold_data, folds, y, k_list)]

## 5. Các bước cho một bộ dữ liệu

`load_dataset` → `run_real` → `run_resamples("null")` (permutation) → `run_resamples("rand")` (random-k) →
`summarize` → `plot` → `save_config`. Mọi trạng thái của một bộ nằm trong đối tượng `D`.

In [ ]:
def load_dataset(name):
    spec = DATASET_SPECS[name]
    D = SimpleNamespace(name=name, branch=spec["branch"], task=spec["task"], k_list=list(spec["k"]),
                        n_perm=spec["n_perm"], n_random=spec["n_random"], ds_dir=os.path.join(DATA_ROOT, name),
                        result_dir=os.path.join(RESULT_ROOT, name), stopped=False, t0=time.time())
    os.makedirs(D.result_dir, exist_ok=True)
    meta = pd.read_csv(os.path.join(D.ds_dir, "metadata.csv"), index_col="sample_id")
    pf = pd.read_csv(os.path.join(D.ds_dir, "perm_folds.csv"), index_col="sample_id")
    meta.index = meta.index.astype(str); pf.index = pf.index.astype(str)
    assert list(pf.index) == list(meta.index) and (pf["label"] == meta["label"]).all()
    D.data_sha256 = {f: sha256(os.path.join(D.ds_dir, f)) for f in ["metadata.csv", "perm_folds.csv"]}
    # Bo chep tu v2 giu config.json goc; thong tin build/fold cua permutation nam o perm_config.json
    cfg = "perm_config.json" if os.path.isfile(os.path.join(D.ds_dir, "perm_config.json")) else "config.json"
    D.data_config = json.load(open(os.path.join(D.ds_dir, cfg), encoding="utf-8"))
    print(f"metadata: {len(meta)} mau |", meta["class"].value_counts().to_dict())

    D.excluded = []
    if D.branch == "2_histology" and EXCLUDE_DIAGNOSIS_MISMATCH and "diagnosis_mismatch" in meta.columns:
        bad = meta["diagnosis_mismatch"].astype(str).str.lower().eq("true")
        D.excluded = meta.index[bad].tolist()
        meta, pf = meta.loc[~bad], pf.loc[~bad]
        print("Bo ca primary_diagnosis trai project:", D.excluded)
    if "sample_type" in meta.columns and D.branch != "3_sanity":
        assert (meta["sample_type"] == "Primary Tumor").all(), "nhanh 1/2 chi duoc co mau u"

    D.y = meta["label"].to_numpy().astype(int)
    D.groups = meta["patient_id"].astype(str).to_numpy()
    D.sample_ids = meta.index.to_numpy()
    pos = pd.Series(np.arange(len(D.sample_ids)), index=D.sample_ids)
    D.class_names = {int(k): v for k, v in meta.groupby("label")["class"].first().items()}
    assert sorted(D.class_names) == [0, 1], f"chi ho tro tac vu 2 lop: {D.class_names}"
    D.chance = 0.5   # BAcc ngau nhien cua tac vu 2 lop

    # Doc folds/fold_<r>/{train,val,test}.h5 (giong feature selection); log2 goc, KHONG chuan hoa.
    # Chi so tr/va/te tro vao D.y/D.groups de lay nhan (that hoac da xao) cua dung tung mau.
    D.folds, D.fold_data, D.genes = [], [], None
    for r in range(1, N_SPLITS + 1):
        idx, mats = [], []
        for s in ["train", "val", "test"]:
            fp = os.path.join(D.ds_dir, "folds", f"fold_{r}", f"{s}.h5")
            h = pd.read_hdf(fp, key="data")
            h.index = h.index.astype(str)
            h = h.loc[~h.index.isin(D.excluded)]
            assert set(h.index) == set(pf.index[pf[f"fold_{r}"] == s]), f"fold {r} {s}: khac perm_folds.csv"
            assert (h["label"].to_numpy() == pf.loc[h.index, "label"].to_numpy()).all(), f"fold {r} {s}: lech nhan"
            g = h.columns.drop("label")
            D.genes = g if D.genes is None else D.genes
            assert g.equals(D.genes), f"fold {r} {s}: danh sach gene khac"
            Xs = np.ascontiguousarray(h[g].to_numpy(), dtype=np.float64)
            assert not np.isnan(Xs).any()
            idx.append(pos.loc[h.index].to_numpy()); mats.append(Xs)
            D.data_sha256[f"folds/fold_{r}/{s}.h5"] = sha256(fp)
        tr, va, te = idx
        assert not (set(D.groups[tr]) & set(D.groups[va]) or set(D.groups[tr]) & set(D.groups[te])
                    or set(D.groups[va]) & set(D.groups[te])), f"fold {r}: trung benh nhan giua cac split"
        for s_idx in idx:
            assert len(np.unique(D.y[s_idx])) == 2, f"fold {r}: thieu lop"
        D.folds.append((tr, va, te)); D.fold_data.append(tuple(mats))
    D.genes = D.genes.to_numpy().astype(str)
    D.n_genes = len(D.genes)
    assert max(D.k_list) < D.n_genes
    test_of = np.zeros(len(D.y), dtype=int)
    for r, (_, _, te) in enumerate(D.folds, 1):
        test_of[te] = r
    assert (test_of > 0).all(), "co mau khong lam test o fold nao"
    print(f"Dung: {len(D.y)} mau x {D.n_genes} gene | {len(set(D.groups))} benh nhan |",
          {D.class_names[c]: int((D.y == c).sum()) for c in (0, 1)})
    print("train/val/test moi fold:", [tuple(len(i) for i in f) for f in D.folds])
    pf.assign(test_fold=test_of).to_csv(os.path.join(D.result_dir, "fold_assignment.csv"))

    D.df = {key: load_prev(D, key) for key in FILES}
    print(f"Da co: {D.df['null'].pid.nunique()} hoan vi | {D.df['rand'].rep.nunique()} lan random-k | "
          f"nhan that: {sorted(D.df['real'].pipeline.unique())}")
    return D


def run_real(D):
    # Nhan that: fit tren train; BAcc tren val (tham khao) va test (chinh); ma tran nham lan (test gop 5 fold),
    # gene duoc chon; do thoi gian 1 lan chay
    done = done_ids(D.df["real"].assign(pid=-1), "pid")
    real_rows, conf_rows, gene_rows, time_rows = [], [], [], []
    for p in [p for p in PIPELINES if -1 not in done.get(p, set())]:
        t0 = time.perf_counter()
        pooled = {(cw, k): np.empty_like(D.y) for cw in CLASS_WEIGHTS for k in D.k_list}
        for f, ((Xtr, Xva, Xte), (tr, va, te)) in enumerate(zip(D.fold_data, D.folds), 1):
            preds = fold_predictions(Xtr, D.y[tr], np.vstack([Xva, Xte]), p, CLASS_WEIGHTS, D.k_list)
            for (cw, k), (pred, g) in preds.items():
                pv, pt = pred[:len(va)], pred[len(va):]
                pooled[(cw, k)][te] = pt
                real_rows += [{"class_weight": cw, "pipeline": p, "k": k, "fold": f, "split": s,
                               "bacc": balanced_accuracy_score(D.y[i], pr)} for s, i, pr in [("val", va, pv),
                                                                                            ("test", te, pt)]]
                gene_rows += [{"class_weight": cw, "pipeline": p, "k": k, "fold": f, "rank": r + 1,
                               "gene": D.genes[j]} for r, j in enumerate(g)]
        for (cw, k), pred in pooled.items():
            cm = confusion_matrix(D.y, pred, labels=[0, 1])
            conf_rows += [{"class_weight": cw, "pipeline": p, "k": k, "true": a, "pred": b, "count": int(cm[a, b])}
                          for a in (0, 1) for b in (0, 1)]
        sec = time.perf_counter() - t0
        time_rows.append({"pipeline": p, "sec_per_run": sec})
        print(f"  {p:10s}: 1 lan chay (moi cau hinh, moi k, 5 fold) = {sec:.1f}s")
    for key, rows in [("real", real_rows), ("conf", conf_rows), ("gene", gene_rows), ("time", time_rows)]:
        append_csv(rows, D, key)
        D.df[key] = load_prev(D, key)
    display(D.df["real"].groupby(["split", "class_weight", "pipeline", "k"])["bacc"].mean().unstack("k").round(3))

    # Uoc luong thoi gian (lan chay tren dung BLAS nhieu luong -> uoc luong hoi lac quan)
    nd, rd = done_ids(D.df["null"], "pid"), done_ids(D.df["rand"], "rep")
    sec = D.df["time"].set_index("pipeline")["sec_per_run"]
    # random-k khong chon gen -> re hon nhieu so voi 1 hoan vi (he so 0.3 la uoc doan)
    est = sum(float(sec[p]) * (max(D.n_perm - len(nd.get(p, set())), 0) if MODE == "permutation"
                               else 0.3 * max(D.n_random - len(rd.get(p, set())), 0))
              for p in PIPELINES) / (N_WORKERS * 0.85)
    print(f"  Uoc luong con lai cho bo nay: ~{est / 3600:.1f} gio (ngan sach con {hours_left():.1f} gio)")


def run_resamples(D, kind):
    # kind = "null" (permutation, id = pid) hoac "rand" (random-k, id = rep); chay theo lo, luu sau moi lo
    id_col, n_total = ("pid", D.n_perm) if kind == "null" else ("rep", D.n_random)
    done = done_ids(D.df[kind], id_col)
    items = [(i, [p for p in PIPELINES if i not in done.get(p, set())]) for i in range(n_total)]
    items = [it for it in items if it[1]]
    batch = N_WORKERS * BATCH_PER_WORKER
    print(f"  {'Permutation' if kind == 'null' else 'Random-k'}: can chay {len(items)} (lo {batch})")
    for s in range(0, len(items), batch):
        lot = items[s:s + batch]
        t0 = time.time()
        chunks = [lot[i::N_WORKERS] for i in range(N_WORKERS) if lot[i::N_WORKERS]]
        res = Parallel(n_jobs=N_JOBS)(delayed(run_chunk)(kind, c, D.fold_data, D.folds, D.y, D.groups, D.k_list)
                                      for c in chunks)
        rows = [r for part in res for r in part]
        append_csv(rows, D, kind)
        D.df[kind] = typed(pd.concat([D.df[kind], pd.DataFrame(rows)], ignore_index=True))
        lot_h = (time.time() - t0) / 3600
        n_done = s + len(lot)
        print(f"    {n_done}/{len(items)} | lo {lot_h * 60:.1f} phut | con ~{lot_h * (len(items) - n_done) / len(lot):.1f} gio"
              f" | ngan sach con {hours_left():.1f} gio", flush=True)
        if n_done < len(items) and hours_left() < 1.5 * lot_h + 0.25:
            D.stopped = True
            print("  DUNG AN TOAN: sap het ngan sach. Phien moi: RESUME_DIRS = [thu muc perm_test_v3 cu].")
            return

## 6. Tổng hợp: p gốc, Holm trong từng họ, quyết định

- **Điểm một lần chạy** = BAcc trên **test**, trung bình qua 5 fold (cùng fold cho nhãn thật, hoán vị và random-k).
- **p_perm** = (#{hoán vị: BAcc ≥ thật} + 1)/(B + 1); **p_random** = (#{random-k ≥ thật} + 1)/(R + 1).
- **Họ kiểm định** = (bộ dữ liệu, tác vụ, `class_weight`, loại kiểm định ∈ {perm, random_k}); mỗi họ có
  3 pipeline × |k| kiểm định; Holm (1979) hiệu chỉnh riêng trong từng họ. `balanced` là kết quả chính,
  `none` là sensitivity analysis — hai cấu hình không gộp chung họ.
- **Độ phân giải:** p nhỏ nhất = 1/(B+1). Với B = R = 1000, p nhỏ nhất ≈ 0,001 < 0,05/21 nên Holm đạt được 0,05
  cho cả họ 15 (nhánh 1) lẫn 21 kiểm định (nhánh 2), ở cả permutation và random-k.
- **Quyết định (đặt trước)**, mỗi tiêu chí có hai cột: hậu tố `_p` dùng p gốc (từng kiểm định riêng lẻ, tham khảo),
  hậu tố `_holm` dùng p sau Holm (**kết luận chính**):
  - *có tín hiệu*: `signal_p` = `p_perm < 0,05`; `signal_holm` = `p_perm_holm < 0,05`;
  - *hơn random-k*: `beats_random_p` = `p_random < 0,05` và `gain_vs_random ≥ MIN_GAIN`; `beats_random_holm` tương tự
    với `p_random_holm`;
  - *chọn gen có ích*: `useful_p` = `signal_p` và `beats_random_p`; `useful_holm` = `signal_holm` và `beats_random_holm`.

  `verdict` (chính) dựa trên các cột `_holm`, `verdict_p` dựa trên các cột `_p`. Khi mới có một
  notebook, `decision.csv` ghi rõ "chưa gộp" — kết luận cuối lấy sau khi gộp (xem đầu notebook). Nhánh (3): đạt sanity nếu BAcc nhãn thật ≥ 0,9.

In [ ]:
def run_means(df, id_col):
    # BAcc trung binh 5 fold cua moi lan chay (chi lay lan chay du 5 fold)
    if not len(df):
        return pd.DataFrame(columns=[id_col, "class_weight", "pipeline", "k", "bacc"])
    g = df.groupby([id_col, "class_weight", "pipeline", "k"])["bacc"].agg(["mean", "size"]).reset_index()
    return g[g["size"] == N_SPLITS].rename(columns={"mean": "bacc"}).drop(columns="size")


def summarize(D):
    rdf = D.df["real"]
    real_mean = rdf[rdf.split == "test"].groupby(["class_weight", "pipeline", "k"])["bacc"].agg(["mean", "std"])
    val_mean = rdf[rdf.split == "val"].groupby(["class_weight", "pipeline", "k"])["bacc"].mean()
    null_m, rand_m = run_means(D.df["null"], "pid"), run_means(D.df["rand"], "rep")
    rows = []
    for (cw, p, k), (real, sd) in real_mean.iterrows():
        null = null_m[(null_m.class_weight == cw) & (null_m.pipeline == p) & (null_m.k == k)]["bacc"].to_numpy()
        rk = rand_m[(rand_m.class_weight == cw) & (rand_m.pipeline == p) & (rand_m.k == k)]["bacc"].to_numpy()
        rows.append({
            "dataset": D.name, "branch": D.branch, "task": D.task, "class_weight": cw,
            "analysis": "main" if cw == MAIN_CW else "sensitivity", "pipeline": p, "k": int(k), "chance": D.chance,
            "bacc_real": real, "bacc_real_sd_folds": sd, "bacc_real_val": val_mean.get((cw, p, k), np.nan),
            "n_perm": len(null),
            "null_mean": null.mean() if len(null) else np.nan,
            "null_sd": null.std(ddof=1) if len(null) > 1 else np.nan,
            "null_q95": np.quantile(null, 0.95) if len(null) else np.nan,
            "p_perm": (np.sum(null >= real) + 1) / (len(null) + 1) if len(null) else np.nan,
            "n_random": len(rk),
            "random_mean": rk.mean() if len(rk) else np.nan,
            "random_q95": np.quantile(rk, 0.95) if len(rk) else np.nan,
            "p_random": (np.sum(rk >= real) + 1) / (len(rk) + 1) if len(rk) else np.nan,
            "gain_vs_null": real - null.mean() if len(null) else np.nan,
            "gain_vs_random": real - rk.mean() if len(rk) else np.nan,
        })
    summary = pd.DataFrame(rows)
    summary["p_perm_holm"] = np.nan
    summary["p_random_holm"] = np.nan

    # Holm rieng trong tung ho (dataset, task, class_weight, loai kiem dinh)
    family_size = len(ALL_PIPELINES) * len(D.k_list)
    parts = []
    for cw in summary.class_weight.unique():
        for test, pcol, ncol, n_target in [("perm", "p_perm", "n_perm", D.n_perm),
                                           ("random_k", "p_random", "n_random", D.n_random)]:
            g = summary[(summary.class_weight == cw) & summary[pcol].notna()]
            if not len(g):
                continue
            complete = (len(g) == family_size) and (g[ncol].min() >= n_target)
            adj = holm(g[pcol])
            summary.loc[g.index, f"{pcol}_holm"] = adj
            parts.append(pd.DataFrame({
                "dataset": D.name, "branch": D.branch, "task": D.task, "class_weight": cw,
                "analysis": "main" if cw == MAIN_CW else "sensitivity", "test": test,
                "family": f"{D.name}|{D.task}|{cw}|{test}", "family_size": len(g),
                "family_size_expected": family_size, "family_complete": complete,
                "pipeline": g.pipeline, "k": g.k, "n_resamples": g[ncol], "p_raw": g[pcol], "p_holm": adj,
                "reject_holm": adj < ALPHA, "seed": SEED, "commit_hash": GIT_COMMIT}))
    pvals = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    pvals.to_csv(os.path.join(D.result_dir, "pvalues_adjusted.csv"), index=False)

    # Moi tieu chi 2 cot: _p (p goc, tham khao) va _holm (p sau Holm, ket luan chinh)
    for suf, pp, pr in [("p", "p_perm", "p_random"), ("holm", "p_perm_holm", "p_random_holm")]:
        summary[f"signal_{suf}"] = summary[pp] < ALPHA
        summary[f"beats_random_{suf}"] = (summary[pr] < ALPHA) & (summary.gain_vs_random >= MIN_GAIN)
        summary[f"useful_{suf}"] = summary[f"signal_{suf}"] & summary[f"beats_random_{suf}"]
    summary["seed"], summary["commit_hash"] = SEED, GIT_COMMIT
    summary.to_csv(os.path.join(D.result_dir, "summary.csv"), index=False)

    decision = summary.groupby(["class_weight", "analysis"]).agg(
        n_tests=("bacc_real", "size"), best_bacc=("bacc_real", "max"), min_p_perm=("p_perm", "min"),
        min_p_perm_holm=("p_perm_holm", "min"), min_p_random=("p_random", "min"),
        min_p_random_holm=("p_random_holm", "min"),
        **{f"n_{c}_{suf}": (f"{c}_{suf}", "sum") for suf in ["p", "holm"]
           for c in ["signal", "beats_random", "useful"]}).reset_index()
    decision.insert(0, "dataset", D.name); decision.insert(1, "branch", D.branch); decision.insert(2, "task", D.task)
    decision["complete"] = bool(len(pvals) == 0 or pvals.family_complete.all()) and not D.stopped
    has_perm, has_rand = summary.n_perm.max() > 0, summary.n_random.max() > 0

    def verdict(suf):
        sig, beat, use = (decision[f"n_{c}_{suf}"] for c in ["signal", "beats_random", "useful"])
        if D.n_perm == 0 and D.n_random == 0:
            return np.where(decision.best_bacc >= 0.9, "SANITY OK: BAcc nhan that >= 0.9",
                            "SANITY CAN XEM LAI: BAcc nhan that < 0.9")
        if has_perm and has_rand:
            return np.where(use > 0, "Co tin hieu va chon gen co ich",
                   np.where(sig > 0, "Co tin hieu nhung chon gen KHONG hon random-k",
                            "KHONG co bang chung ve tin hieu"))
        if has_perm:
            return np.where(sig > 0, "Co tin hieu (chua gop random-k)",
                            "KHONG co bang chung ve tin hieu (chua gop random-k)")
        return np.where(beat > 0, "Chon gen hon random-k (chua gop permutation)",
                        "Chon gen KHONG hon random-k (chua gop permutation)")

    decision["verdict"] = verdict("holm")     # ket luan chinh
    decision["verdict_p"] = verdict("p")      # tham khao, chua hieu chinh da kiem dinh
    decision.to_csv(os.path.join(D.result_dir, "decision.csv"), index=False)
    if not decision.complete.all():
        print("  CANH BAO: ho kiem dinh chua du (thieu pipeline hoac chua du hoan vi) -> p hieu chinh CHUA phai ket qua cuoi.")
    display(decision)
    cols = ["class_weight", "pipeline", "k", "bacc_real", "bacc_real_val", "null_mean", "p_perm", "p_perm_holm",
            "random_mean", "p_random", "p_random_holm", "gain_vs_random",
            "signal_p", "signal_holm", "beats_random_p", "beats_random_holm", "useful_p", "useful_holm"]
    display(summary[cols].round(4))
    return summary, pvals, decision

In [ ]:
import matplotlib.pyplot as plt

PIPE_NAMES = {"ftest_lr": "F-test + LR", "svm_rfe": "SVM-RFE + SVM", "pam": "PAM"}


def plot(D, summary):
    fig, axes = plt.subplots(len(CLASS_WEIGHTS), len(ALL_PIPELINES), figsize=(4 * len(ALL_PIPELINES),
                             3.2 * len(CLASS_WEIGHTS)), squeeze=False, sharey=True)
    for i, cw in enumerate(CLASS_WEIGHTS):
        for j, p in enumerate(ALL_PIPELINES):
            ax = axes[i][j]
            g = summary[(summary.class_weight == cw) & (summary.pipeline == p)].sort_values("k")
            if len(g):
                x = np.arange(len(g))
                if g.n_perm.max() > 1:
                    ax.fill_between(x, g.null_mean - 1.96 * g.null_sd, g.null_mean + 1.96 * g.null_sd, alpha=0.25,
                                    label="Null 95%")
                if g.n_random.max() > 0:
                    ax.plot(x, g.random_mean, "s--", label="Random-k TB")
                ax.plot(x, g.bacc_real, "o-", color="k", label="Nhãn thật")
                for xi, (_, r) in zip(x, g.iterrows()):
                    mark = "*" if r.signal_holm else "+" if r.signal_p else ""
                    if mark:
                        ax.annotate(mark, (xi, r.bacc_real), ha="center", va="bottom", fontsize=14)
                ax.set_xticks(x); ax.set_xticklabels(g.k)
            ax.axhline(D.chance, color="gray", ls=":", lw=1)
            ax.set_title(f"{PIPE_NAMES[p]} · {cw}" + (" (chính)" if cw == MAIN_CW else " (sensitivity)"), fontsize=9)
            if j == 0:
                ax.set_ylabel("BAcc (TB 5 fold)")
            if i == len(CLASS_WEIGHTS) - 1:
                ax.set_xlabel("k (số gene)")
    axes[0][0].legend(fontsize=7)
    fig.suptitle(f"{D.name} · {D.task}: BAcc thật vs null vs random-k (* = p_perm Holm < {ALPHA}; + = chi p_perm goc < {ALPHA})")
    plt.tight_layout()
    plt.savefig(os.path.join(D.result_dir, "perm_test_v3_bacc.png"), dpi=150)
    plt.show()

    if summary.p_perm_holm.notna().any():
        fig, axes = plt.subplots(1, len(CLASS_WEIGHTS), figsize=(4.5 * len(CLASS_WEIGHTS), 3.2), squeeze=False)
        for ax, cw in zip(axes[0], CLASS_WEIGHTS):
            h = summary[summary.class_weight == cw].pivot_table(index="pipeline", columns="k", values="p_perm_holm")
            h = h.reindex([p for p in ALL_PIPELINES if p in h.index])
            im = ax.imshow(-np.log10(h.values), cmap="viridis", vmin=0, vmax=3)
            ax.set_xticks(range(h.shape[1])); ax.set_xticklabels(h.columns)
            ax.set_yticks(range(h.shape[0])); ax.set_yticklabels([PIPE_NAMES[p] for p in h.index])
            for a in range(h.shape[0]):
                for b in range(h.shape[1]):
                    ax.text(b, a, f"{h.values[a, b]:.3f}", ha="center", va="center", fontsize=7,
                            color="w" if h.values[a, b] > 0.01 else "k")
            ax.set_title(f"{cw}: p_perm Holm"); ax.set_xlabel("k")
        fig.colorbar(im, ax=axes[0].tolist(), label="-log10(p_holm)")
        plt.savefig(os.path.join(D.result_dir, "perm_test_v3_pvalues.png"), dpi=150, bbox_inches="tight")
        plt.show()


def save_config(D):
    config = {
        "mode": MODE, "dataset": D.name, "branch": D.branch, "task": D.task, "k_list": D.k_list, "seed": SEED,
        "commit_hash": GIT_COMMIT,
        "runs": [{"dataset": D.name, "task": D.task, "k": k, "seed": SEED, "commit_hash": GIT_COMMIT} for k in D.k_list],
        "data_dir": D.ds_dir, "data_sha256": D.data_sha256,
        "data_build": {"script": D.data_config.get("script") or D.data_config.get("notebook"),
                       "commit_hash": D.data_config.get("commit_hash")},
        "class_names": D.class_names, "n_samples": int(len(D.y)), "n_patients": int(len(set(D.groups))),
        "class_counts": {D.class_names[c]: int((D.y == c).sum()) for c in (0, 1)}, "n_genes": int(D.n_genes),
        "excluded_samples": D.excluded, "scaling": "none (raw log2)",
        "pipelines_this_run": PIPELINES, "all_pipelines": ALL_PIPELINES,
        "class_weights": CLASS_WEIGHTS, "main_class_weight": MAIN_CW,
        "pam_prior": {"balanced": "equal", "none": "class proportions"},
        "lr_c": LR_C, "svm_c": SVM_C, "n_splits": N_SPLITS,
        "fold_method": "perm_folds.csv cua bo du lieu: " + str(D.data_config.get("perm_folds", {}).get("method")),
        "fit_split": "train", "eval_split": "test (p-value); val chi ghi lai (bacc_real_val)",
        "fold_sizes_train_val_test": [[int(len(i)) for i in f] for f in D.folds],
        "n_permutations": D.n_perm, "n_random_k": D.n_random,
        "permutation_seed": "np.random.default_rng([SEED, pid]), xao theo benh nhan",
        "random_k_seed": "np.random.default_rng([SEED, 7, rep])",
        "multiple_testing": "Holm trong tung ho (dataset, task, class_weight, test in {perm, random_k})",
        "alpha": ALPHA, "min_gain": MIN_GAIN,
        "n_perm_done": int(D.df["null"].pid.nunique()), "n_random_done": int(D.df["rand"].rep.nunique()),
        "resume_dirs": RESUME_DIRS, "stopped_early": D.stopped,
        "platform": "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else platform.node(), "cpu_count": os.cpu_count(),
        "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "numpy": np.__version__,
                     "pandas": pd.__version__},
        "elapsed_hours_this_dataset": (time.time() - D.t0) / 3600,
    }
    with open(os.path.join(D.result_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, indent=1, ensure_ascii=False)

## 7. Chạy tất cả

Chạy lần lượt các bộ trong `RUN_DATASETS`. Bộ nào đã đủ hoán vị/random-k (từ `RESUME_DIRS`) thì chỉ tổng hợp lại.
Hết ngân sách thời gian → dừng sau lô đang chạy, các bộ còn lại để phiên sau.

In [ ]:
import gc

ALL = {"summary": [], "pvals": [], "decision": []}
status = {}
for name in RUN_DATASETS:
    print("\n" + "=" * 100 + f"\n{name}  ({DATASET_SPECS[name]['branch']} | {DATASET_SPECS[name]['task']})"
          f" | ngan sach con {hours_left():.1f} gio\n" + "=" * 100)
    D = load_dataset(name)
    run_real(D)
    kind, n_target, id_col = (("null", D.n_perm, "pid") if MODE == "permutation" else ("rand", D.n_random, "rep"))
    if hours_left() > 0.25:
        run_resamples(D, kind)
    D.stopped = D.stopped or n_target > D.df[kind][id_col].nunique()
    summary, pvals, decision = summarize(D)
    plot(D, summary)
    save_config(D)
    ALL["summary"].append(summary); ALL["pvals"].append(pvals); ALL["decision"].append(decision)
    status[name] = "CHUA XONG" if D.stopped else "xong"
    print(f"{name}: {status[name]} | {(time.time() - D.t0) / 60:.1f} phut")
    del D; gc.collect()
    if hours_left() < 0.25:
        print("Het ngan sach thoi gian -> dung; cac bo con lai chay o phien sau (RESUME_DIRS).")
        break

for key, fn in [("summary", "all_summary.csv"), ("pvals", "all_pvalues_adjusted.csv"),
                ("decision", "all_decisions.csv")]:
    frames = [f for f in ALL[key] if len(f)]
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(os.path.join(RESULT_ROOT, fn), index=False)
for name in RUN_DATASETS:
    status.setdefault(name, "CHUA CHAY")
json.dump({"mode": MODE, "status": status, "commit_hash": GIT_COMMIT, "seed": SEED, "run_datasets": RUN_DATASETS,
           "elapsed_hours": (time.time() - T_START) / 3600}, open(os.path.join(RESULT_ROOT, "run_status.json"), "w"),
          indent=1)
display(pd.concat(ALL["decision"], ignore_index=True) if ALL["decision"] else "Chua co bo nao xong")
print("Trang thai:", status)

zip_path = shutil.make_archive(os.path.join(OUT_ROOT, "perm_test_v3_random_k"), "zip", OUT_ROOT, "perm_test_v3_random_k")
print("Da nen:", zip_path, f"| tong thoi gian {(time.time() - T_START) / 3600:.2f} gio")